# Data Contract and Structural Validation

This notebook provides a reusable first pass for a tabular prediction project. It contains no data and no saved outputs.

## Objective

Before modeling, verify file layout, schema consistency, key uniqueness, and record order. Update the configuration below so it matches the local dataset.

In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

data_dir = project_root / "data"
parquet_pattern = "**/*.parquet"

# Adapt these names to the local dataset.
key_columns = ["timestamp", "entity_id"]
order_columns = ["timestamp"]
target_column = "target"
weight_column = "sample_weight"


In [ ]:
parquet_files = sorted(data_dir.glob(parquet_pattern))
if not parquet_files:
    raise FileNotFoundError(
        f"No Parquet files found under {data_dir}. "
        "Add local data and update the path configuration."
    )

print(f"Found {len(parquet_files)} Parquet files")
for file_path in parquet_files:
    print(file_path.relative_to(project_root))


In [ ]:
inventory = []

for file_path in parquet_files:
    parquet_file = pq.ParquetFile(file_path)
    metadata = parquet_file.metadata

    inventory.append(
        {
            "file": file_path.name,
            "rows": metadata.num_rows,
            "row_groups": metadata.num_row_groups,
            "columns": metadata.num_columns,
            "size_gib": round(file_path.stat().st_size / (1024**3), 3),
        }
    )

inventory_df = pd.DataFrame(inventory)
inventory_df


In [ ]:
reference_schema = pq.ParquetFile(parquet_files[0]).schema_arrow

schema_checks = pd.DataFrame(
    {
        "file": [file_path.name for file_path in parquet_files],
        "schema_matches_reference": [
            pq.ParquetFile(file_path).schema_arrow.equals(reference_schema)
            for file_path in parquet_files
        ],
    }
)

schema_checks


In [ ]:
dtype_summary_df = (
    pd.DataFrame(
        {
            "column": [field.name for field in reference_schema],
            "dtype": [str(field.type) for field in reference_schema],
        }
    )
    .groupby("dtype", as_index=False)
    .agg(columns=("column", "count"))
)

dtype_summary_df["percentage"] = (
    dtype_summary_df["columns"] / len(reference_schema) * 100
).round(2)

dtype_summary_df


In [ ]:
available_key_columns = [
    column for column in key_columns if column in reference_schema.names
]

if len(available_key_columns) != len(key_columns):
    raise KeyError(
        "Update key_columns so every configured key exists in the dataset schema."
    )

duplicate_checks = []
for file_path in parquet_files:
    key_data = pd.read_parquet(file_path, columns=available_key_columns)
    duplicate_checks.append(
        {
            "file": file_path.name,
            "rows": len(key_data),
            "duplicate_keys": key_data.duplicated(
                subset=available_key_columns
            ).sum(),
        }
    )

duplicate_checks_df = pd.DataFrame(duplicate_checks)
duplicate_checks_df


In [ ]:
available_order_columns = [
    column for column in order_columns if column in reference_schema.names
]

if len(available_order_columns) != len(order_columns):
    raise KeyError(
        "Update order_columns so every configured column exists in the dataset schema."
    )

order_checks = []
for file_path in parquet_files:
    order_data = pd.read_parquet(file_path, columns=available_order_columns)
    order_index = pd.MultiIndex.from_frame(order_data)
    order_checks.append(
        {
            "file": file_path.name,
            "is_monotonic_increasing": order_index.is_monotonic_increasing,
        }
    )

order_checks_df = pd.DataFrame(order_checks)
order_checks_df


## Interpretation Checklist

- Are all partitions structurally compatible?
- Do configured key columns uniquely identify a record?
- Is the stored record order consistent with the intended validation timeline?
- Can later analysis read selected columns and partitions instead of the entire dataset?

Record the answers in project documentation before moving to exploratory analysis or modeling.